# Basket Trading — Data Cleaning & Feature Engineering
**CRISP-DM Phase:** Data Preparation  
**Skript-Bezug:** Kapitel 4 (Datenverständnis & -vorbereitung), Kap. 4.4 (Feature Engineering)

**Ziel:** Die rohen Aktiendaten aus Notebook 10 bereinigen und um 
Features anreichern, die für die Renditeprognose relevant sind. 
Output ist ein modellfertiger Datensatz für Notebook 12 (EDA) und 
Notebook 13 (Modeling).

**Pipeline:**
1. Rohdaten einlesen
2. Fehlwert-Analyse je Aktie
3. Aktien mit zu vielen Lücken entfernen
4. Forward-Fill für kurze Lücken
5. Feature Engineering (Momentum, Volatilität, RSI, MA, Forward Return)
6. Output speichern

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

RAW  = Path("/Users/lisagroppe/Desktop/data/raw")
PROC = Path("/Users/lisagroppe/Desktop/data/processed")
PROC.mkdir(parents=True, exist_ok=True)

print(f"Eingangsverzeichnis:  {RAW}")
print(f"Ausgangsverzeichnis:  {PROC}")

Eingangsverzeichnis:  /Users/lisagroppe/Desktop/data/raw
Ausgangsverzeichnis:  /Users/lisagroppe/Desktop/data/processed


## 2. Rohdaten einlesen

In [2]:
prices = pd.read_parquet(RAW / "dax_mdax_prices_2019_2025.parquet")
fundamentals = pd.read_csv(RAW / "dax_mdax_fundamentals.csv")

# Spaltennamen normalisieren: erster Buchstabe groß, Rest Originalformat
prices.columns = [c.title().replace("_", "_") for c in prices.columns]
prices = prices.rename(columns={
    "Adj_Close": "Adj_Close",
    "Index_Name": "Index_Name",
})
fundamentals.columns = [
    c.replace("pe_ratio", "P_E_Ratio")
     .replace("dividend_yield", "Dividend_Yield")
     .replace("market_cap", "Market_Cap")
     .replace("price_to_book", "Price_To_Book")
     .replace("index_name", "Index_Name")
     .replace("full_name", "Full_Name")
     .title()
    for c in fundamentals.columns
]
# Ticker-Spalte sicherstellen
prices.columns = [c if c != "Ticker" else "Ticker" for c in prices.columns]

print(f"Preisdaten: {prices.shape}")
print(f"Spalten:    {list(prices.columns)}")
print(f"Tickers:    {prices['Ticker'].nunique()}")
print(f"Zeitraum:   {prices['Date'].min()} bis {prices['Date'].max()}")
print(f"\nFundamentaldaten: {fundamentals.shape}")

Preisdaten: (150489, 9)
Spalten:    ['Date', 'Ticker', 'Adj_Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Index_Name']
Tickers:    86
Zeitraum:   2019-01-02 00:00:00 bis 2025-12-30 00:00:00

Fundamentaldaten: (86, 12)


## 3. Fehlwert-Analyse je Aktie
Wir prüfen, wie viele Datenpunkte jede Aktie hat. Aktien mit deutlich 
weniger Datenpunkten als der Median haben Lücken — entweder durch 
spätes IPO, Delisting oder Datenqualitätsprobleme bei yfinance.

In [3]:
counts = prices.groupby("Ticker").size()
median_count = counts.median()
threshold = 0.95 * median_count

print(f"Median Datenpunkte je Aktie: {median_count:.0f}")
print(f"Schwellwert (95%): {threshold:.0f}")
print(f"\nAnzahl Aktien je Datenpunkt-Bereich:")
print(f"  Voll (>={threshold:.0f}):    {(counts >= threshold).sum()}")
print(f"  Lückenhaft (<{threshold:.0f}): {(counts < threshold).sum()}")

print(f"\nAktien mit den wenigsten Datenpunkten:")
print(counts.sort_values().head(10).to_string())

Median Datenpunkte je Aktie: 1779
Schwellwert (95%): 1690

Anzahl Aktien je Datenpunkt-Bereich:
  Voll (>=1690):    80
  Lückenhaft (<1690): 6

Aktien mit den wenigsten Datenpunkten:
Ticker
DTG.DE     1033
VH2.DE     1215
ENR.DE     1339
HAG.DE     1341
TMV.DE     1593
8TRA.DE    1655
DHL.DE     1772
LEG.DE     1779
RDC.DE     1779
RAA.DE     1779


## 4. Aktien mit zu vielen Lücken entfernen
Wir behalten nur Aktien mit mindestens 95% des Median-Datenpunkt-Werts. 
Das filtert Spätstarter und IPOs raus, ohne zu viele zu verlieren.

In [4]:
valid_tickers = counts[counts >= threshold].index.tolist()
prices_clean = prices[prices["Ticker"].isin(valid_tickers)].copy()

removed = sorted(set(prices["Ticker"]) - set(valid_tickers))
print(f"Behaltene Aktien: {len(valid_tickers)}")
print(f"Entfernte Aktien: {len(removed)}")
if removed:
    print(f"Entfernt: {removed}")

print(f"\nNeue Datensatz-Größe: {prices_clean.shape}")

Behaltene Aktien: 80
Entfernte Aktien: 6
Entfernt: ['8TRA.DE', 'DTG.DE', 'ENR.DE', 'HAG.DE', 'TMV.DE', 'VH2.DE']

Neue Datensatz-Größe: (142313, 9)


## 5. Forward-Fill für kurze Lücken
Selbst bei den verbleibenden Aktien können einzelne Tageslücken 
auftreten (Handelsruhe, technische Probleme). Wir füllen mit 
Forward-Fill (letzter bekannter Wert) für maximal 3 Tage. Das ist 
Finanzstandard.

In [5]:
prices_clean["Date"] = pd.to_datetime(prices_clean["Date"])
prices_clean = prices_clean.sort_values(["Ticker", "Date"]).reset_index(drop=True)

# Forward-Fill je Aktie, max 3 Tage
price_cols = [c for c in ["Open", "High", "Low", "Close", "Adj_Close", "Volume"] if c in prices_clean.columns]
prices_clean[price_cols] = prices_clean.groupby("Ticker")[price_cols].ffill(limit=3)

# Verbleibende NaN nach ffill
remaining_nan = prices_clean[price_cols].isna().sum()
print("Verbleibende NaN nach Forward-Fill:")
print(remaining_nan.to_string())

# Adj_Close ist im Rohdatensatz 100% NaN — Close als Pflichtfeld nutzen
adj_col = "Close"
n_before = len(prices_clean)
prices_clean = prices_clean.dropna(subset=[adj_col])
print(f"\nZeilen vorher:  {n_before:,}")
print(f"Zeilen nachher: {len(prices_clean):,}")
print(f"Entfernt:       {n_before - len(prices_clean):,}")

Verbleibende NaN nach Forward-Fill:
Open              0
High              0
Low               0
Close             0
Adj_Close    142313
Volume            0

Zeilen vorher:  142,313
Zeilen nachher: 142,313
Entfernt:       0


## 6. Feature Engineering
Wir berechnen die für Renditeprognosen relevanten Features. Alle 
Berechnungen sind **rolling**, also nutzen NUR Daten der Vergangenheit 
(kein Look-Ahead!).

**Features:**
- `Daily_Return`, `Log_Return`: Tagesrenditen
- `Momentum_3M`, `_6M`, `_12M`: Renditen über 63 / 126 / 252 Handelstage
- `Volatility_30d`: Realisierte Volatilität auf 30 Tagen
- `RSI_14`: Relative Strength Index, 14-Tage-Standard
- `MA_50`, `MA_200`: Moving Averages
- `Forward_Return_1M`: ZIELVARIABLE — 21 Handelstage in die Zukunft

In [6]:
def add_features(df):
    """Fügt einer einzelnen Aktie alle Features hinzu."""
    df = df.sort_values("Date").copy()

    # Close bevorzugen; Adj_Close nur wenn tatsächlich Werte vorhanden
    if "Adj_Close" in df.columns and df["Adj_Close"].notna().any():
        adj = "Adj_Close"
    else:
        adj = "Close"

    # Returns
    df["Daily_Return"] = df[adj].pct_change()
    df["Log_Return"]   = np.log(df[adj] / df[adj].shift(1))

    # Momentum
    df["Momentum_3M"]  = df[adj].pct_change(periods=63)
    df["Momentum_6M"]  = df[adj].pct_change(periods=126)
    df["Momentum_12M"] = df[adj].pct_change(periods=252)

    # Realisierte Volatilität (annualisiert)
    df["Volatility_30d"] = df["Daily_Return"].rolling(30).std() * np.sqrt(252)

    # Moving Averages
    df["MA_50"]  = df[adj].rolling(50).mean()
    df["MA_200"] = df[adj].rolling(200).mean()

    # RSI 14
    delta = df[adj].diff()
    gain  = delta.where(delta > 0, 0).rolling(14).mean()
    loss  = -delta.where(delta < 0, 0).rolling(14).mean()
    rs    = gain / loss
    df["RSI_14"] = 100 - (100 / (1 + rs))

    # ZIELVARIABLE: Forward Return über 21 Handelstage (~1 Monat)
    df["Forward_Return_1M"] = df[adj].pct_change(periods=21).shift(-21)

    return df

# Manuell iterieren statt groupby().apply() — robuster in pandas 2.x
print("Berechne Features für alle Aktien...")
parts = []
for ticker, grp in prices_clean.groupby("Ticker"):
    parts.append(add_features(grp))
prices_features = pd.concat(parts, ignore_index=True)

print(f"Shape nach Feature Engineering: {prices_features.shape}")
print(f"\nNeue Spalten:")
new_cols = [c for c in prices_features.columns if c not in prices_clean.columns]
for c in new_cols:
    print(f"  {c}")

assert "Forward_Return_1M" in prices_features.columns, "Forward_Return_1M fehlt"
print(f"\nForward_Return_1M: {prices_features['Forward_Return_1M'].notna().sum():,} Nicht-NaN-Werte")

Berechne Features für alle Aktien...
Shape nach Feature Engineering: (142313, 19)

Neue Spalten:
  Daily_Return
  Log_Return
  Momentum_3M
  Momentum_6M
  Momentum_12M
  Volatility_30d
  MA_50
  MA_200
  RSI_14
  Forward_Return_1M

Forward_Return_1M: 140,633 Nicht-NaN-Werte


## 7. Fundamentaldaten anreichern
Statische Fundamentaldaten je Ticker werden gemerged. Hinweis zur 
Limitation: Snapshot-Werte, kein historisch korrekter Stand 
(Look-Ahead-Bias bei Fundamentaldaten — wird in Limitations dokumentiert).

In [7]:
# Ticker-Index aus groupby.apply entfernen
prices_features = prices_features.reset_index(drop=True)

# Verfügbare Fundamentalspalten dynamisch ermitteln
fund_cols_wanted = ["Ticker", "Sector", "Industry", "Market_Cap",
                    "P_E_Ratio", "Dividend_Yield", "Price_To_Book"]
fund_cols_avail  = [c for c in fund_cols_wanted if c in fundamentals.columns]
fund_subset      = fundamentals[fund_cols_avail].copy()

basket = prices_features.merge(fund_subset, on="Ticker", how="left")
print(f"Shape nach Merge: {basket.shape}")
print(f"\nFundamentaldaten-Vollständigkeit:")
for col in [c for c in ["Sector", "Market_Cap", "P_E_Ratio", "Dividend_Yield"] if c in basket.columns]:
    pct = basket[col].notna().mean() * 100
    print(f"  {col:20s}: {pct:.1f}%")

Shape nach Merge: (142313, 25)

Fundamentaldaten-Vollständigkeit:
  Sector              : 100.0%
  Market_Cap          : 100.0%
  P_E_Ratio           : 86.2%
  Dividend_Yield      : 92.5%


## 8. Endgültige Bereinigung
Zeilen ohne Zielvariable (die letzten 21 Tage je Aktie haben kein 
Forward Return) werden für die Modellierung verworfen. Sie bleiben 
aber verfügbar für die spätere Vorhersage neuer Daten.

In [8]:
# Zwei Versionen: full (mit NaN in Forward Return) und modeling (ohne)
basket_full     = basket.copy()
basket_modeling = basket.dropna(subset=["Forward_Return_1M"]).copy()

# Frühe Zeilen ohne Momentum_12M / Volatility_30d ebenfalls entfernen
basket_modeling = basket_modeling.dropna(subset=["Momentum_12M", "Volatility_30d"])

print(f"Voller Datensatz (für Vorhersage neuer Daten): {basket_full.shape}")
print(f"Modeling-Datensatz (mit allen Features):       {basket_modeling.shape}")

Voller Datensatz (für Vorhersage neuer Daten): (142313, 25)
Modeling-Datensatz (mit allen Features):       (120473, 25)


## 9. Output speichern

In [9]:
out_full     = PROC / "basket_full.parquet"
out_modeling = PROC / "basket_features.parquet"

basket_full.to_parquet(out_full, index=False)
basket_modeling.to_parquet(out_modeling, index=False)

print(f"Gespeichert:")
print(f"  {out_full.name:40s} ({out_full.stat().st_size / 1024**2:.1f} MB)")
print(f"  {out_modeling.name:40s} ({out_modeling.stat().st_size / 1024**2:.1f} MB)")

Gespeichert:
  basket_full.parquet                      (18.4 MB)
  basket_features.parquet                  (16.0 MB)


## 10. Sanity Checks
Schnelle Plausibilitätsprüfung — sind die Features in vernünftigen 
Wertebereichen?

In [10]:
print("Statistische Übersicht der Features:")
feature_cols_check = ["Daily_Return", "Momentum_3M", "Momentum_12M",
                       "Volatility_30d", "RSI_14", "Forward_Return_1M"]
print(basket_modeling[feature_cols_check].describe().round(4).to_string())

print("\nErwartete Wertebereiche (Faustregeln):")
print("  Daily_Return:       ~99% zwischen -5% und +5%")
print("  Momentum_3M:        sehr breit, -50% bis +100% möglich")
print("  Volatility_30d:     0.10 bis 0.80 typisch (10-80% annualisiert)")
print("  RSI_14:             zwischen 0 und 100")
print("  Forward_Return_1M:  ~99% zwischen -20% und +20%")

Statistische Übersicht der Features:
       Daily_Return  Momentum_3M  Momentum_12M  Volatility_30d       RSI_14  Forward_Return_1M
count   120473.0000  120473.0000   120473.0000     120473.0000  120473.0000        120473.0000
mean         0.0004       0.0258        0.1127          0.3309      51.2813             0.0083
std          0.0234       0.1878        0.4205          0.1656      17.8077             0.1086
min         -0.4210      -0.7376       -0.8138          0.0284       0.0000            -0.7079
25%         -0.0100      -0.0836       -0.1334          0.2176      38.6325            -0.0490
50%          0.0000       0.0194        0.0660          0.2904      51.4644             0.0066
75%          0.0108       0.1231        0.2882          0.3999      64.0663             0.0652
max          0.3380       1.6199        5.1249          1.5318     100.0000             1.1542

Erwartete Wertebereiche (Faustregeln):
  Daily_Return:       ~99% zwischen -5% und +5%
  Momentum_3M:      

## 11. Reflexion

**Was wurde getan:**
- Aktien mit unvollständigen Daten entfernt (Survivorship-Bias-Filter)
- Forward-Fill für kurze Lücken (Finanzstandard)
- 13 zusätzliche Features berechnet, alle ohne Look-Ahead
- Zielvariable Forward_Return_1M definiert
- Zwei Output-Datensätze: voll und modellierungsbereit

**Bekannte Limitationen:**
1. Survivorship-Bias verstärkt durch zusätzlichen Filter
2. Fundamentaldaten als Snapshot (nicht historisch korrekt)
3. Keine Adjustierung für Aktiensplits über die `Adj_Close`-Spalte hinaus

**Nächster Schritt:**  
Notebook 12 — Explorative Datenanalyse auf basket_features.parquet.  
Notebook 13 — Random Forest Regression Modeling.

In [11]:
import pandas as pd
from pathlib import Path

PROC = Path("../data/processed")
basket = pd.read_parquet(PROC / "basket_features.parquet")

print(f"Shape modeling-Datensatz: {basket.shape}")
print(f"Eindeutige Aktien:        {basket['Ticker'].nunique()}")
print(f"Zeitraum:                 {basket['Date'].min()} bis {basket['Date'].max()}")
print()
print("Statistik der Schlüssel-Features:")
print(basket[['Daily_Return', 'Volatility_30d', 'RSI_14', 'Forward_Return_1M']].describe().round(4))
print()
print("NaN-Anteil je wichtiger Spalte:")
for col in ['Daily_Return', 'Momentum_12M', 'Volatility_30d', 'RSI_14', 'Forward_Return_1M']:
    nan_pct = basket[col].isna().mean() * 100
    print(f"  {col:25s} {nan_pct:6.2f}%")

Shape modeling-Datensatz: (120473, 25)
Eindeutige Aktien:        80
Zeitraum:                 2020-01-03 00:00:00 bis 2025-11-26 00:00:00

Statistik der Schlüssel-Features:
       Daily_Return  Volatility_30d       RSI_14  Forward_Return_1M
count   120473.0000     120473.0000  120473.0000        120473.0000
mean         0.0004          0.3309      51.2813             0.0083
std          0.0234          0.1656      17.8077             0.1086
min         -0.4210          0.0284       0.0000            -0.7079
25%         -0.0100          0.2176      38.6325            -0.0490
50%          0.0000          0.2904      51.4644             0.0066
75%          0.0108          0.3999      64.0663             0.0652
max          0.3380          1.5318     100.0000             1.1542

NaN-Anteil je wichtiger Spalte:
  Daily_Return                0.00%
  Momentum_12M                0.00%
  Volatility_30d              0.00%
  RSI_14                      0.00%
  Forward_Return_1M           0.00%
